# Gold — fact_sales
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Grain** | One row per transaction |
| **Sources** | `silver.transactions` + `silver.orders` + `silver.customers` |
| **Target** | `{catalog}.gold.fact_sales` |
| **Pattern** | Join Silver tables → MERGE into Gold |
| **Depends on** | All 3 Silver tasks must complete first |

**What this builds:**

```
fact_sales
├── transaction_id   — grain key
├── order_id         — degenerate dimension
├── customer_id      — degenerate dimension
├── order_date       — from silver.orders
├── status           — from silver.orders
├── payment_mode     — from silver.transactions
├── quantity         ← fact measure 1
├── total_amount     ← fact measure 2
├── _amount_flag     — carry-through DQ signal from Silver
└── _loaded_at       — audit: when this row landed in Gold
```

Same notebook works for both batch 1 and batch 2 — MERGE handles inserts and updates identically.

## Setup — Widgets & Constants

Widget values are overridden at runtime by Workflow job parameters.

In [ ]:
dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'silver')
dbutils.widgets.text('target_schema', 'gold')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')

SILVER_TXN       = f'{CATALOG}.{SOURCE_SCHEMA}.transactions'
SILVER_ORDERS    = f'{CATALOG}.{SOURCE_SCHEMA}.orders'
SILVER_CUSTOMERS = f'{CATALOG}.{SOURCE_SCHEMA}.customers'
TABLE            = f'{CATALOG}.{TARGET_SCHEMA}.fact_sales'

print(f'Sources : {SILVER_TXN}')
print(f'          {SILVER_ORDERS}')
print(f'          {SILVER_CUSTOMERS}')
print(f'Target  : {TABLE}')

## Step 1 — Read Silver Tables

Transactions drive the grain — one fact row per transaction.
Orders and customers are joined to enrich each row with context.

In [ ]:
txn    = spark.table(SILVER_TXN).alias('txn')
ord_df = spark.table(SILVER_ORDERS).alias('ord')
cust   = spark.table(SILVER_CUSTOMERS).alias('cust')

print(f'silver.transactions rows : {txn.count()}')
print(f'silver.orders rows       : {ord_df.count()}')
print(f'silver.customers rows    : {cust.count()}')

## Step 2 — Build Fact Rows

Join chain:
1. `transactions` → `orders` on `order_id` (inner — every transaction must have an order)
2. `orders` → `customers` on `customer_id` (left — keep the transaction even if customer is missing)

Aliases on all three tables prevent column ambiguity.

In [ ]:
from pyspark.sql.functions import col, current_timestamp

fact_df = txn \
    .join(ord_df, col('txn.order_id')    == col('ord.order_id'),    'inner') \
    .join(cust,   col('ord.customer_id') == col('cust.customer_id'), 'left') \
    .select(
        col('txn.transaction_id'),
        col('txn.order_id'),
        col('ord.customer_id'),
        col('ord.order_date'),
        col('ord.status'),
        col('txn.payment_mode'),
        col('txn.quantity'),
        col('txn.total_amount'),
        col('txn._amount_flag'),
        current_timestamp().alias('_loaded_at')
    )

print(f'Fact rows built: {fact_df.count()}')
fact_df.display()

## Step 3 — Create Gold Table (first run only)

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}')

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE} (
        transaction_id STRING,
        order_id       STRING,
        customer_id    STRING,
        order_date     DATE,
        status         STRING,
        payment_mode   STRING,
        quantity       INT,
        total_amount   DOUBLE,
        _amount_flag   STRING,
        _loaded_at     TIMESTAMP
    )
    USING DELTA
''')

print(f'Table ready: {TABLE}')

## Step 4 — MERGE into fact_sales

Merge on `transaction_id`.
- **Match** → update all columns (picks up status changes on linked orders)
- **No match** → insert new row

Same code runs on batch 1 and batch 2 without any changes.

In [ ]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, TABLE)

target.alias('t').merge(
    fact_df.alias('s'),
    't.transaction_id = s.transaction_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print('MERGE complete')

## Step 5 — Verify

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

result = spark.table(TABLE)
print(f'Total rows in {TABLE}: {result.count()}')
print(f'Amount-flagged rows  : {result.filter(col("_amount_flag").isNotNull()).count()}')

print('\n--- Revenue by Status ---')
result.groupBy('status') \
      .agg(spark_round(spark_sum('total_amount'), 2).alias('total_revenue')) \
      .orderBy('status').display()

print('\n--- Revenue by Payment Mode ---')
result.groupBy('payment_mode') \
      .agg(spark_round(spark_sum('total_amount'), 2).alias('total_revenue')) \
      .orderBy('payment_mode').display()

result.display()